# 🚀 Fine-Tuning Qwen3-VL 8B with 4-bit QLoRA on Google Colab (Free T4 GPU)
### Floor Plan Structured Analysis (Rooms, Length, Width, Doors, Windows)

This notebook fine-tunes **Qwen3-VL 8B** (or Qwen2.5-VL 7B) on your blended CubiCasa5k + Synthetic Floor Plan dataset.

**Hardware Target:** Google Colab Free Tier (NVIDIA Tesla T4 - 15GB VRAM)
**Memory Optimizations:**
- **4-bit NormalFloat Quantization (NF4 via Unsloth / bitsandbytes):** Model weights occupy ~5.2 GB instead of 16 GB.
- **Vision Token Scaling (`max_pixels = 768 * 768`):** Prevents token count explosions on large floor plan blueprints.
- **LoRA (Rank 16, Alpha 32):** Adapts attention and MLP projection layers with vision encoder frozen.
- **Gradient Checkpointing + `paged_adamw_8bit`:** Offloads activation spikes cleanly.
- **Google Drive Persistence:** Checkpoints save directly to your Google Drive to prevent loss from Colab timeouts.

In [ ]:
# Step 1: Verify GPU Allocation
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Please switch your Colab runtime to GPU! (Runtime -> Change runtime type -> T4 GPU)"
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\nConnected to GPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)")

In [ ]:
# Step 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/floorplan_reader_project'
OUTPUT_MODEL_DIR = f'{PROJECT_DIR}/models/qwen3_vl_floorplan_lora'
import os
os.makedirs(OUTPUT_MODEL_DIR, exist_ok=True)
print(f"Model checkpoints will be saved to: {OUTPUT_MODEL_DIR}")

In [ ]:
# Step 3: Install Unsloth and Training Dependencies
# Unsloth provides 2x faster training and 70% memory reduction on Colab T4
!pip install -q --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q unsloth_zoo trl peft accelerate bitsandbytes datasets xformers torchvision cairosvg svglib pydantic

In [ ]:
# Step 4: Configure Model and LoRA Parameters
MODEL_ID = 'unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit'
FALLBACK_MODEL_ID = 'unsloth/Qwen2.5-VL-7B-Instruct-unsloth-bnb-4bit'
LORA_TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
]
print(f'Ready to fine-tune: {MODEL_ID}')


### Step 5: Load Qwen3-VL 8B in 4-bit with Unsloth
Using `FastVisionModel.from_pretrained` loads the model directly in 4-bit NormalFloat (NF4), using only ~5.2 GB of VRAM.

In [ ]:
import torch
from unsloth import FastVisionModel

MODEL_ID = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"

try:
    print(f"Loading {MODEL_ID} in 4-bit via Unsloth...")
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name=MODEL_ID,
        load_in_4bit=True,
        use_gradient_checkpointing="unsloth",
    )
except Exception as e:
    print(f"Loading {MODEL_ID} failed ({e}). Loading fallback Qwen2.5-VL-7B...")
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name="unsloth/Qwen2.5-VL-7B-Instruct-unsloth-bnb-4bit",
        load_in_4bit=True,
        use_gradient_checkpointing="unsloth",
    )

# Step 6: Apply LoRA Adapters
print("Applying LoRA configuration...")
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,  # Keep vision encoder frozen to stay safely under 15GB T4 VRAM
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    random_state=42,
)
model.print_trainable_parameters()

### Step 7: Load Dataset and Configure Data Collator

In [ ]:
from datasets import load_dataset
from unsloth import UnslothVisionDataCollator

train_jsonl_path = f"{PROJECT_DIR}/processed/train.jsonl"
val_jsonl_path = f"{PROJECT_DIR}/processed/val.jsonl"

data_files = {"train": train_jsonl_path}
if os.path.exists(val_jsonl_path):
    data_files["val"] = val_jsonl_path

raw_datasets = load_dataset("json", data_files=data_files)
train_dataset = raw_datasets["train"]
eval_dataset = raw_datasets.get("val")

print(f"Loaded {len(train_dataset)} training floor plans.")
if eval_dataset:
    print(f"Loaded {len(eval_dataset)} validation floor plans.")

### Step 8: Configure SFTTrainer and Begin Training
We use an effective batch size of 8 (`batch_size=1`, `gradient_accumulation_steps=8`) with `paged_adamw_8bit`.

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_MODEL_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,  # Effective batch size = 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    optim="paged_adamw_8bit",
    fp16=True,
    logging_steps=5,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    max_seq_length=2048,
    gradient_checkpointing=True,
    dataset_text_field="",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
)

print("Starting LoRA fine-tuning on Colab T4...")
trainer_stats = trainer.train()
print("Fine-tuning complete!")

### Step 9: Save Checkpoints to Google Drive
Saves the fine-tuned LoRA adapter and tokenizer so you can load it anytime for inference without re-training.

In [ ]:
print(f"Saving LoRA adapter to {OUTPUT_MODEL_DIR}...")
model.save_pretrained(OUTPUT_MODEL_DIR)
tokenizer.save_pretrained(OUTPUT_MODEL_DIR)
print(f"Model saved successfully to Google Drive: {OUTPUT_MODEL_DIR}")

# Optional: Push adapter to Hugging Face Hub (Uncomment if desired)
# model.push_to_hub("your-hf-username/qwen3-vl-floorplan-reader", token="your_hf_token")
# tokenizer.push_to_hub("your-hf-username/qwen3-vl-floorplan-reader", token="your_hf_token")

### Step 10: Quick Test Prediction on a Floor Plan

In [ ]:
FastVisionModel.for_inference(model)
import json, re
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# Test on first validation image
test_sample = eval_dataset[0] if eval_dataset else train_dataset[0]
test_img_path = test_sample['messages'][0]['content'][0]['image']
test_img = Image.open(test_img_path).convert('RGB')
w, h = test_img.size

prompt = (
    'Analyze this architectural floor plan drawing. Detect all rooms, doors, and windows. '
    'Output a structured JSON object containing "rooms", "doors", and "windows" '
    'with normalized [ymin, xmin, ymax, xmax] box_2d (0-1000 scale).'
)

messages = [
    {'role': 'user', 'content': [{'type': 'image', 'image': test_img}, {'type': 'text', 'text': prompt}]}
]

input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(images=[test_img], text=[input_text], return_tensors='pt').to('cuda')

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=1024, temperature=0.1)

gen_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('Raw Generation:\n', gen_text)

# Extract JSON
match = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', gen_text) or re.search(r'\{[\s\S]*\}', gen_text)
if match:
    clean_json = match.group(1) if '```' in gen_text else match.group(0)
    try:
        data = json.loads(clean_json)
        fig, ax = plt.subplots(figsize=(10, 8))
        ax.imshow(test_img)
        for r in data.get('rooms', []):
            box = r.get('box_2d', [])
            if len(box) == 4:
                ymin, xmin, ymax, xmax = [v / 1000.0 for v in box]
                rect = patches.Rectangle((xmin * w, ymin * h), (xmax - xmin) * w, (ymax - ymin) * h,
                                         linewidth=2, edgecolor='blue', facecolor='none')
                ax.add_patch(rect)
                ax.text(xmin * w + 5, ymin * h + 15, r.get('name', 'room'), color='blue', fontsize=10, weight='bold',
                        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))
        plt.axis('off')
        plt.title(f"Detected {len(data.get('rooms', []))} Rooms")
        plt.show()
    except Exception as err:
        print('JSON parsing notice:', err)
